In [2]:
import json
import pandas as pd

# Menampilkan semua baris/kolom jika diperlukan saat inspeksi
pd.set_option('display.max_columns', None)

print("ok")

ok


In [3]:
# 1. Buka dan baca file JSON
with open('dataset_from_500.json', 'r', encoding='utf-8') as file:
    raw_data = json.load(file)

# 2. Siapkan list kosong untuk menampung baris data
flattened_data = []

# 3. Bongkar struktur JSON (Flattening)
for user in raw_data['data']:
    steam_id = user['steam_id']
    for game in user['library']:
        flattened_data.append({
            'steam_id': steam_id,
            'app_id': game['app_id'],
            'name': game['name'],
            'playtime': game['playtime']
        })

# 4. Konversi ke DataFrame Pandas
df = pd.DataFrame(flattened_data)

print(f"Bentuk awal DataFrame: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head() # Menampilkan 5 baris pertama untuk dicek

Bentuk awal DataFrame: 33236 baris, 4 kolom


,steam_id,app_id,name,playtime
0,76561198418191535,10,Counter-Strike,1012
1,76561198418191535,70,Half-Life,95
2,76561198418191535,80,Counter-Strike: Condition Zero,309
3,76561198418191535,220,Half-Life 2,700
4,76561198418191535,240,Counter-Strike: Source,147


In [7]:
import re

# TAHAP 2: KEYWORD FILTERING DENGAN REGEX
blacklist_keywords = [
    'wallpaper', 'dedicated server', 'server', 'software', 'sdk', 
    'tool', 'playtest', 'demo', 'soundtrack', 'editor', 
    'benchmark', 'application', 'viewer', 'creator'
]

# Gabungkan kata kunci menjadi pola Regex dengan batas kata (\b)
# Contoh hasil pattern: \b(wallpaper|dedicated server|server|software|...)\b
regex_pattern = r'\b(?:' + '|'.join(blacklist_keywords) + r')\b'

# Gunakan pandas str.contains dengan regex=True
# mask_dropped akan bernilai True JIKA nama game mengandung kata blacklist yang berdiri sendiri
mask_dropped = df['name'].str.contains(regex_pattern, case=False, regex=True, na=False)

# mask_valid adalah kebalikannya (yang tidak mengandung blacklist)
mask_valid = ~mask_dropped

# Pisahkan data
df_filtered = df[mask_valid].copy()
df_dropped = df[mask_dropped].copy()

# Cek selisih baris
print(f"Total baris SEBELUM difilter : {df.shape[0]}")
print(f"Total baris yang TERHAPUS    : {df_dropped.shape[0]}")
print(f"Total baris SESUDAH difilter : {df_filtered.shape[0]}")

print("\n--- Contoh Aplikasi/Game yang Masuk Blacklist ---")
for name in df_dropped['name'].unique()[:15]:
    print(f"- {name}")

print("\n" + "="*50 + "\n")

# TAHAP 3: EKSTRAKSI GAME MAPPING
# Ambil pasangan app_id dan name yang unik
game_mapping_df = df_filtered[['app_id', 'name']].drop_duplicates(subset=['app_id'])

# Ubah ke bentuk dictionary dan simpan ke JSON
game_mapping_dict = dict(zip(game_mapping_df['app_id'], game_mapping_df['name']))
with open('game_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(game_mapping_dict, f, indent=4)
print("Kamus metadata 'game_mapping.json' berhasil dibuat!")

# Buang kolom 'name' dari tabel utama (menyisakan steam_id, app_id, playtime)
df_clean = df_filtered.drop(columns=['name'])

print("\nTampilan DataFrame bersih (siap untuk re-validasi):")
display(df_clean.head())

Total baris SEBELUM difilter : 33236
Total baris yang TERHAPUS    : 294
Total baris SESUDAH difilter : 32942

--- Contoh Aplikasi/Game yang Masuk Blacklist ---
- Wallpaper Engine
- H1Z1: Test Server
- Hunt: Showdown 1896 (Test Server)
- Tom Clancy's Rainbow Six Siege - Test Server
- Movavi Video Editor Plus 2020
- Fluid Engine PC Live Wallpaper
- Gecata by Movavi 5 - Game Recording Software


Kamus metadata 'game_mapping.json' berhasil dibuat!

Tampilan DataFrame bersih (siap untuk re-validasi):


,steam_id,app_id,playtime
0,76561198418191535,10,1012
1,76561198418191535,70,95
2,76561198418191535,80,309
3,76561198418191535,220,700
4,76561198418191535,240,147
